In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
df=spark.read.format('parquet')\
   .option('path','abfss://bronze@datalakecommerce.dfs.core.windows.net/customers')\
    .load()

### **Deleting _rescued_data column**

In [0]:
df=df.drop(col("_rescued_data"))

### **Adding Domain Column**

In [0]:
df=df.withColumn('domain',split(col("email"),'@')[1])

### **Adding Full Name Column**

In [0]:
df=df.withColumn('full_name',concat(col("first_name"),lit(' '),col("last_name")))

In [0]:
df=df.drop("first_name","last_name")

### **Data Writing**

In [0]:
df.write.format('delta').mode('overwrite')\
    .save('abfss://silver@datalakecommerce.dfs.core.windows.net/customers')

### **Registration table in Unity Catalog**

In [0]:
%sql
create table if not exists ecommerce.silver.customers
using delta
location 'abfss://silver@datalakecommerce.dfs.core.windows.net/customers'